# Chapter 34 — Reinforcement Learning II: Policy Iteration, SARSA, Deep RL & Actor-Critic

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part5_production_rl/ch34_reinforcement_learning_ii/ch34_notebook.ipynb)

Companion notebook for **Chapter 34** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- Policy iteration alternates evaluating the current policy and improving it, and usually converges in very few rounds.
- Reinforcement learning is needed when the transition model is unknown and labeled examples could never cover the state space.
- Model-based agents plan with a model; model-free agents learn Q-values (Q-learning) or a policy (policy search) directly.
- TD learning updates after every step using the next state's estimate, so it needs no model and no waiting for the episode to end.
- SARSA is on-policy and plays safe near danger; Q-learning is off-policy and learns the optimal path even while exploring.
- When there are too many states for a table, a neural network approximates the values: that is deep RL, and a DQN has one output per action.
- Policy gradients and actor-critic methods improve the policy directly; AlphaGo combines policy and value networks with Monte Carlo Tree Search.

## Example 1: Policy Iteration on the 4x3 Grid World

This solves the same grid world as Chapter 33, but with policy iteration. It starts from a bad policy, 'always go up', and repeats two steps: work out the exact value of every state under the current policy by solving a set of linear equations, then switch each state to its best action. After just five rounds the policy stops changing. Look at the last table: the utilities match the value-iteration answers from Chapter 33, so two different methods agree on the same optimal policy.

In [1]:
# Chapter 34 - Example 1: Policy Iteration on the 4x3 Grid World
import numpy as np

print("=== Policy Iteration on the 4x3 Grid World ===\n")
print("Same world as Chapter 33: wall at (2,2), +1 at (4,3), -1 at (4,2),")
print("step reward -0.04, action succeeds with p=0.8, slips sideways 0.1 each.\n")
print("Policy iteration alternates two steps until the policy stops changing:")
print("  1. Policy EVALUATION : work out U(s) for the CURRENT policy (no max)")
print("  2. Policy IMPROVEMENT: in each state, switch to the best action\n")

rows, cols = 3, 4
WALL = (2, 2)
TERMINALS = {(4, 3): 1.0, (4, 2): -1.0}
R_STEP, GAMMA = -0.04, 1.0
ACTIONS = {"Up": (0, 1), "Down": (0, -1), "Left": (-1, 0), "Right": (1, 0)}
SIDE = {"Up": ["Left", "Right"], "Down": ["Left", "Right"],
        "Left": ["Up", "Down"], "Right": ["Up", "Down"]}
ARROW = {"Up": "^", "Down": "v", "Left": "<", "Right": ">"}

states = [(c, r) for c in range(1, cols + 1) for r in range(1, rows + 1) if (c, r) != WALL]
nonterm = [s for s in states if s not in TERMINALS]
idx = {s: i for i, s in enumerate(nonterm)}

def move(s, a):
    dc, dr = ACTIONS[a]
    ns = (s[0] + dc, s[1] + dr)
    if ns == WALL or not (1 <= ns[0] <= cols and 1 <= ns[1] <= rows):
        return s
    return ns

def outcomes(s, a):
    return [(0.8, move(s, a))] + [(0.1, move(s, b)) for b in SIDE[a]]

def evaluate(policy):
    """Policy evaluation: solve the LINEAR system U = R + gamma * P_pi U exactly."""
    n = len(nonterm)
    A = np.eye(n)
    b = np.zeros(n)
    for s in nonterm:
        i = idx[s]
        for p, ns in outcomes(s, policy[s]):
            b[i] += p * R_STEP
            if ns in TERMINALS:
                b[i] += p * GAMMA * TERMINALS[ns]
            else:
                A[i, idx[ns]] -= p * GAMMA
    U_vec = np.linalg.solve(A, b)
    U = {s: U_vec[idx[s]] for s in nonterm}
    U.update(TERMINALS)
    return U

def q_value(U, s, a):
    return sum(p * (R_STEP + GAMMA * U[ns]) for p, ns in outcomes(s, a))

def show(policy):
    for r in range(rows, 0, -1):
        line = f"    row {r} |"
        for c in range(1, cols + 1):
            s = (c, r)
            if s == WALL: line += "  # "
            elif s in TERMINALS: line += f" {'+1' if TERMINALS[s] > 0 else '-1'} "
            else: line += f"  {ARROW[policy[s]]} "
        print(line)

# Start from a deliberately poor policy: always go Up
policy = {s: "Up" for s in nonterm}
print("Initial policy (always Up):")
show(policy)

for it in range(1, 20):
    U = evaluate(policy)                                   # step 1
    changed = 0
    for s in nonterm:                                      # step 2
        best = max(ACTIONS, key=lambda a: q_value(U, s, a))
        if q_value(U, s, best) > q_value(U, s, policy[s]) + 1e-12:
            policy[s] = best
            changed += 1
    print(f"\nIteration {it}: U(1,1)={U[(1,1)]:+.4f}  U(3,3)={U[(3,3)]:+.4f}  "
          f"actions changed = {changed}")
    if changed == 0:
        print("Policy is stable -> it is optimal. Stop.")
        break

U = evaluate(policy)
print("\nFinal optimal policy:")
show(policy)

print("\nFinal utilities compared with VALUE iteration from Chapter 33:")
vi = {(1, 1): 0.7053, (1, 3): 0.8116, (3, 3): 0.9178, (4, 1): 0.3878}
print(f"    {'state':<8}{'policy iter.':>14}{'value iter.':>13}")
for s, v in vi.items():
    print(f"    {str(s):<8}{U[s]:>14.4f}{v:>13.4f}")

print("\nBoth methods reach the same answer. Policy iteration needed only a")
print("handful of iterations because each one solves the evaluation exactly;")
print("value iteration needed 22 smaller steps to creep to the same values.")

=== Policy Iteration on the 4x3 Grid World ===

Same world as Chapter 33: wall at (2,2), +1 at (4,3), -1 at (4,2),
step reward -0.04, action succeeds with p=0.8, slips sideways 0.1 each.

Policy iteration alternates two steps until the policy stops changing:
  1. Policy EVALUATION : work out U(s) for the CURRENT policy (no max)
  2. Policy IMPROVEMENT: in each state, switch to the best action

Initial policy (always Up):
    row 3 |  ^   ^   ^  +1 
    row 2 |  ^   #   ^  -1 
    row 1 |  ^   ^   ^   ^ 

Iteration 1: U(1,1)=-1.4662  U(3,3)=-0.2000  actions changed = 6

Iteration 2: U(1,1)=+0.5082  U(3,3)=+0.9178  actions changed = 1

Iteration 3: U(1,1)=+0.6910  U(3,3)=+0.9178  actions changed = 1

Iteration 4: U(1,1)=+0.7053  U(3,3)=+0.9178  actions changed = 1

Iteration 5: U(1,1)=+0.7053  U(3,3)=+0.9178  actions changed = 0
Policy is stable -> it is optimal. Stop.

Final optimal policy:
    row 3 |  >   >   >  +1 
    row 2 |  ^   #   ^  -1 
    row 1 |  ^   <   <   < 

Final utilit

## Example 2: SARSA vs Q-Learning on Cliff Walking

This is the classic example that shows why on-policy and off-policy learning behave differently. Both agents learn to cross a grid with a cliff along the bottom, while still exploring randomly 10% of the time. Q-learning finds the shortest route, right along the cliff edge. SARSA finds a longer route along the top. Now check the reward table: SARSA earns about -27 per episode against Q-learning's -52, because Q-learning's random steps keep pushing it off the cliff. Shortest is not the same as safest.

In [2]:
# Chapter 34 - Example 2: SARSA (on-policy) vs Q-learning (off-policy) on Cliff Walking
import numpy as np

print("=== SARSA vs Q-Learning: The Cliff-Walking Problem ===\n")
print("A 4x12 grid. Start S is bottom-left, goal G is bottom-right.")
print("The cells between them on the bottom row are a CLIFF.")
print("Every step costs -1. Stepping into the cliff costs -100 and sends you")
print("back to S. Both agents explore with epsilon-greedy (eps = 0.1).\n")
print("    . . . . . . . . . . . .")
print("    . . . . . . . . . . . .")
print("    . . . . . . . . . . . .")
print("    S C C C C C C C C C C G      C = cliff\n")

ROWS, COLS = 4, 12
START, GOAL = (3, 0), (3, 11)
MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]          # up, down, left, right
NAMES = ["^", "v", "<", ">"]
EPS, ALPHA, GAMMA, EPISODES, RUNS = 0.1, 0.5, 1.0, 500, 20

def step(s, a):
    r, c = s[0] + MOVES[a][0], s[1] + MOVES[a][1]
    r, c = min(max(r, 0), ROWS - 1), min(max(c, 0), COLS - 1)
    if r == 3 and 1 <= c <= 10:                      # fell off the cliff
        return START, -100, False
    return (r, c), -1, (r, c) == GOAL

def choose(Q, s, rng):
    if rng.random() < EPS:
        return int(rng.integers(4))                  # explore
    q = Q[s]
    return int(rng.choice(np.flatnonzero(q == q.max())))   # exploit (break ties randomly)

def train(method, seed):
    rng = np.random.default_rng(seed)
    Q = np.zeros((ROWS, COLS, 4))
    returns = []
    for _ in range(EPISODES):
        s, total = START, 0
        a = choose(Q, s, rng)
        for _ in range(500):
            ns, r, done = step(s, a)
            total += r
            na = choose(Q, ns, rng)
            if method == "sarsa":
                target = r + (0 if done else GAMMA * Q[ns][na])        # the action it WILL take
            else:
                target = r + (0 if done else GAMMA * Q[ns].max())      # the BEST action
            Q[s][a] += ALPHA * (target - Q[s][a])
            s, a = ns, na
            if done:
                break
        returns.append(total)
    return Q, np.array(returns)

def greedy_path(Q):
    s, path = START, [START]
    for _ in range(40):
        s, _, done = step(s, int(np.argmax(Q[s])))
        path.append(s)
        if done:
            break
    return path

def draw(path):
    grid = [["." for _ in range(COLS)] for _ in range(ROWS)]
    for c in range(1, 11):
        grid[3][c] = "C"
    for (r, c) in path:
        grid[r][c] = "*"
    grid[3][0], grid[3][11] = "S", "G"
    for row in grid:
        print("    " + " ".join(row))

results = {}
for method in ["sarsa", "qlearning"]:
    all_returns = []
    for seed in range(RUNS):
        Q, ret = train(method, seed)
        all_returns.append(ret)
    results[method] = (Q, np.mean(all_returns, axis=0))

print(f"Average reward per episode during training ({RUNS} runs averaged):\n")
print(f"  {'episodes':<12}{'SARSA':>10}{'Q-learning':>14}")
print("  " + "-" * 36)
for lo, hi in [(0, 50), (50, 100), (100, 200), (200, 350), (350, 500)]:
    s = results["sarsa"][1][lo:hi].mean()
    q = results["qlearning"][1][lo:hi].mean()
    print(f"  {lo+1:>3}-{hi:<8}{s:>10.1f}{q:>14.1f}")

for method, label in [("sarsa", "SARSA"), ("qlearning", "Q-learning")]:
    path = greedy_path(results[method][0])
    print(f"\n{label} learned path ({len(path) - 1} steps):")
    draw(path)

print("\nWhat this shows:")
print("  Q-learning learns the SHORTEST path, right along the cliff edge. Its")
print("  update uses max Q(s',a'), the value of the best action, as if it will")
print("  never explore again. That is OFF-policy learning.")
print("  SARSA learns a SAFER path that keeps well away from the edge. Its update uses the")
print("  action it will actually take next, random exploration included, so it")
print("  knows a random step near the edge can cost -100. That is ON-policy.")
print("  So while both are still exploring, SARSA earns MORE reward per episode")
print("  even though Q-learning's final route is shorter.")

=== SARSA vs Q-Learning: The Cliff-Walking Problem ===

A 4x12 grid. Start S is bottom-left, goal G is bottom-right.
The cells between them on the bottom row are a CLIFF.
Every step costs -1. Stepping into the cliff costs -100 and sends you
back to S. Both agents explore with epsilon-greedy (eps = 0.1).

    . . . . . . . . . . . .
    . . . . . . . . . . . .
    . . . . . . . . . . . .
    S C C C C C C C C C C G      C = cliff

Average reward per episode during training (20 runs averaged):

  episodes         SARSA    Q-learning
  ------------------------------------
    1-50          -109.2        -120.2
   51-100          -34.3         -52.6
  101-200          -29.1         -46.1
  201-350          -27.2         -50.5
  351-500          -26.9         -51.7

SARSA learned path (17 steps):
    * * * * * * * * * * * *
    * . . . . . . . . . . *
    * . . . . . . . . . . *
    S C C C C C C C C C C G

Q-learning learned path (13 steps):
    . . . . . . . . . . . .
    . . . . . . . . 

## Example 3: Policy Gradient by the Finite-Difference Method

This follows the finite-difference recipe exactly. A simple policy starts out preferring the wrong direction in a corridor, so its total reward J is negative. Each iteration it tries a dozen small random changes to its weights, measures how the reward moved, fits a gradient with the regression formula (D^T D)^-1 D^T y, and steps uphill. Without ever learning a value function, J climbs from -0.75 to +0.84. The closing note explains how actor-critic methods speed this up.

In [3]:
# Chapter 34 - Example 3: Policy Gradient by the Finite-Difference Method
import numpy as np
rng = np.random.default_rng(0)

print("=== Policy Gradient: the Finite-Difference Method ===\n")
print("Value-based methods learn U(s) or Q(s,a) and then pick actions from it.")
print("Policy gradient methods skip that: they adjust the policy's parameters")
print("theta DIRECTLY, in the direction that raises the total reward J.\n")
print("The finite-difference recipe:")
print("  1. Nudge the weights m times by small random amounts  delta_theta_i")
print("  2. Run a roll-out of H moves with each nudged policy, measure delta_J_i")
print("  3. Stack nudges as rows of D and reward changes into y, then solve")
print("        grad J = (D^T D)^-1 D^T y        (the linear-regression formula)")
print("  4. Gradient ASCENT:  theta <- theta + eta * grad J\n")

# ---- A corridor: states 0..6, start at 3. Reaching 6 pays +1, reaching 0 costs -1.
N, START, H, GAMMA = 7, 3, 20, 0.95

def features(s):
    return np.array([1.0, (s - 3) / 3.0])           # bias + position

def p_right(theta, s):
    return 1.0 / (1.0 + np.exp(-features(s) @ theta))

def rollout_return(theta):
    s, J = START, 0.0
    for t in range(H):
        s += 1 if rng.random() < p_right(theta, s) else -1
        r = 1.0 if s == N - 1 else (-1.0 if s == 0 else -0.01)
        J += (GAMMA ** t) * r
        if s in (0, N - 1):
            break
    return J

def J_estimate(theta, n=300):
    return np.mean([rollout_return(theta) for _ in range(n)])

print("Environment: a corridor of 7 cells, start in the middle (cell 3).")
print("Right end = +1, left end = -1, every other step = -0.01, horizon H=20.")
print("Policy: P(move right | s) = sigmoid(theta_0 + theta_1 * position)\n")

theta = np.array([-1.0, 0.0])                   # starts out preferring LEFT (wrong way)
m, sigma, eta = 12, 0.3, 0.8

print(f"{'iter':>5}{'theta_0':>10}{'theta_1':>10}{'P(right) at start':>20}{'J':>10}")
print("-" * 57)
for it in range(13):
    J0 = J_estimate(theta)
    if it % 2 == 0 or it == 12:
        print(f"{it:>5}{theta[0]:>10.3f}{theta[1]:>10.3f}{p_right(theta, START):>20.3f}{J0:>10.3f}")
    D = sigma * rng.standard_normal((m, 2))            # m perturbations (rows of D)
    y = np.array([J_estimate(theta + d) - J0 for d in D])
    grad = np.linalg.solve(D.T @ D, D.T @ y)           # (D^T D)^-1 D^T y
    theta = theta + eta * grad

print("\nWhat happened:")
print("  The policy began by preferring the WRONG direction (P(right) = 0.27),")
print("  so it usually fell off the left end and J was negative.")
print("  Nobody told it which way to go. Each iteration it just tried small")
print("  random changes, saw which ones earned more reward, fitted a gradient,")
print("  and stepped uphill. P(right) climbed to nearly 0.9 and J turned positive.\n")

print("=== From here to Actor-Critic ===")
print("  Finite differences need many roll-outs per step, which is slow.")
print("  Actor-critic methods use TWO networks instead:")
print("    the ACTOR  (policy network) chooses actions,")
print("    the CRITIC (Q-network) scores them using the TD error")
print("        delta_t = R_t + gamma*Q(s_t+1, a_t+1) - Q(s_t, a_t).")
print("  The critic's score tells the actor which way to move, one step at a time,")
print("  without waiting for whole roll-outs. AlphaGo and walking robots use this idea.")

=== Policy Gradient: the Finite-Difference Method ===

Value-based methods learn U(s) or Q(s,a) and then pick actions from it.
Policy gradient methods skip that: they adjust the policy's parameters
theta DIRECTLY, in the direction that raises the total reward J.

The finite-difference recipe:
  1. Nudge the weights m times by small random amounts  delta_theta_i
  2. Run a roll-out of H moves with each nudged policy, measure delta_J_i
  3. Stack nudges as rows of D and reward changes into y, then solve
        grad J = (D^T D)^-1 D^T y        (the linear-regression formula)
  4. Gradient ASCENT:  theta <- theta + eta * grad J

Environment: a corridor of 7 cells, start in the middle (cell 3).
Right end = +1, left end = -1, every other step = -0.01, horizon H=20.
Policy: P(move right | s) = sigmoid(theta_0 + theta_1 * position)

 iter   theta_0   theta_1   P(right) at start         J
---------------------------------------------------------
    0    -1.000     0.000               0.269   